# Module 8 Instructor Colab Notebook: Introduction to Multivariable Functions

## A Mathematical Bridge to Artificial Intelligence

A modern AI model does not tune one number. It may contain thousands, millions, or billions of adjustable parameters. Multivariable calculus provides the language for describing how an output or loss changes when many inputs or parameters are present.

This notebook begins with functions of several variables, develops partial derivatives and the gradient, and then connects these ideas to computation graphs, backpropagation, and gradient descent in a parameter space. Every new term is introduced geometrically, algebraically, and through an AI example.

**Recommended pacing:** three class meetings. The Hessian classification section may be treated as an extension while retaining the main gradient storyline.


> **How to use this notebook in class**
>
> Keep three representations visible: a formula, a surface or contour plot, and a numerical parameter vector. Before running each visualization, ask students which direction should increase the function most rapidly. In backpropagation examples, label every local derivative and check dimensions before multiplying or adding paths.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Functions with several inputs | Feature vectors and parameter vectors |
| 2 | Partial derivatives | Sensitivity to one weight at a time |
| 3 | Gradient and directional derivatives | Direction of steepest local increase |
| 4 | Multivariable chain rule and Jacobians | Backpropagation with branching paths |
| 5 | Critical points and Hessian preview | Minima, maxima, and saddle points |
| 6 | Vector gradient descent | Simultaneous training of many weights |
| Integrated lab | Two-parameter linear regression | Optimize slope and intercept together |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable instructor-solution helper.
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


## Module learning objectives

By the end of the module, students should be able to:

1. interpret scalar-valued and vector-valued multivariable functions;
2. identify domains, surfaces, and level curves;
3. compute first and mixed partial derivatives;
4. construct and interpret a gradient vector;
5. compute a directional derivative in a unit direction;
6. form a tangent-plane linearization;
7. apply the multivariable chain rule through multiple paths;
8. interpret a Jacobian and a Hessian; and
9. perform vector gradient-descent updates.

## Prerequisite checkpoint

Students should know vectors, dot products, derivatives, the chain rule, and one-variable gradient descent from earlier modules.


---

# Section 1: Functions with Multiple Inputs

## 1. From one input to many inputs

A one-variable scalar function has the form

$$
f:\mathbb{R}\to\mathbb{R}.
$$

A scalar-valued function of $$n$$ variables has the form

$$
f:\mathbb{R}^n\to\mathbb{R}.
$$

It receives an input vector

$$
\mathbf{x}=
\begin{bmatrix}
x_1\\
x_2\\
\vdots\\
x_n
\end{bmatrix}
$$

and produces one numerical output $$f(\mathbf{x}).$$ A vector-valued function instead has the form

$$
F:\mathbb{R}^n\to\mathbb{R}^m,
$$

so it produces an output vector with $$m$$ components.


## 2. Inputs, features, and parameters

In AI, the input coordinates are often called **features**. A feature vector might contain numerical descriptions such as age, income, temperature, pixel intensity, or embedding coordinates. An **embedding** is a learned numerical vector representation designed so that useful relationships among objects can be expressed through vector geometry.

A model can also be viewed as a function of its parameters. For example,

$$
\widehat{y}=w_1x_1+w_2x_2+b
$$

has input features $$x_1,x_2$$ and adjustable parameters $$w_1,w_2,b.$$ Once the data are fixed, the loss becomes a multivariable function of the parameter vector

$$
\boldsymbol{\theta}=
\begin{bmatrix}
w_1\\
w_2\\
b
\end{bmatrix}.
$$

The set of all possible parameter vectors is called **parameter space**.


## 3. Domain and range in several variables

The **domain** is the set of allowed input vectors. Consider

$$
f(x,y)=\sqrt{9-x^2-y^2}.
$$

The square-root input must be nonnegative:

$$
9-x^2-y^2\geq 0.
$$

Equivalently,

$$
x^2+y^2\leq 9.
$$

Thus the domain is the filled disk of radius three centered at the origin. The range is

$$
0\leq f(x,y)\leq 3.
$$

A multivariable domain is a region rather than merely an interval.


## 4. Evaluating a multivariable function

Let

$$
f(x,y)=x^2+3xy+2y^2.
$$

At the point

$$
(x,y)=(1,2),
$$

substitute both coordinates:

$$
f(1,2)=1^2+3(1)(2)+2(2^2).
$$

Compute term by term:

$$
1^2=1,
$$

$$
3(1)(2)=6,
$$

and

$$
2(2^2)=2(4)=8.
$$

Therefore

$$
\boxed{f(1,2)=1+6+8=15.}
$$


## 5. Surfaces and level curves

The graph of a scalar function $$z=f(x,y)$$ is a surface in three-dimensional space. A **level curve** fixes the output at a constant value:

$$
f(x,y)=c.
$$

Level curves are horizontal slices of the surface viewed from above. On a geographic map, contour lines connect equal elevations. On a loss landscape, they connect parameter pairs with equal loss.

Closely spaced level curves indicate rapid change, while widely spaced curves indicate slower change, assuming equal changes in contour value.


In [ ]:
x_surface = np.linspace(-2.5, 2.5, 160)
y_surface = np.linspace(-2.5, 2.5, 160)
X_surface, Y_surface = np.meshgrid(x_surface, y_surface)
Z_surface = X_surface**2 + 0.5*Y_surface**2

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(X_surface, Y_surface, Z_surface, cmap="viridis", alpha=0.9)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("f(x,y)")
ax1.set_title("Surface view")
ax2 = fig.add_subplot(1, 2, 2)
contours_surface = ax2.contour(X_surface, Y_surface, Z_surface, levels=12, cmap="viridis")
ax2.clabel(contours_surface, inline=True, fontsize=8)
ax2.set_aspect("equal")
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_title("Level-curve view")
plt.tight_layout()
plt.show()


## 6. Vector-valued model outputs

A three-class classifier may produce three scores:

$$
F(\mathbf{x})=
\begin{bmatrix}
s_1(\mathbf{x})\\
s_2(\mathbf{x})\\
s_3(\mathbf{x})
\end{bmatrix}.
$$

Such a function maps an input vector to a score vector. A later softmax operation can convert the scores into probabilities.

The distinction is important:

- a loss is usually scalar-valued because training needs one objective;
- a hidden layer is vector-valued because it produces many activations; and
- a batch of examples is often arranged into a matrix, but the model rule still acts on individual feature vectors or the entire batch in parallel.


> **Common mistakes in Section 1**
>
> - Confusing the number of input coordinates with the number of outputs.
> - Treating a parameter as an observed feature.
> - Describing a multivariable domain as though it must be an interval.
> - Confusing a surface with one of its level curves.
> - Assuming every AI model output is scalar.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Multivariable function | Function receiving two or more input coordinates |
| Scalar-valued function | Function producing one number |
| Vector-valued function | Function producing a vector |
| Feature | Numerical input coordinate used by a model |
| Embedding | Learned vector representation encoding useful object relationships |
| Parameter | Adjustable model quantity learned during training |
| Parameter space | Set of all allowed parameter vectors |
| Surface | Three-dimensional graph of a two-input scalar function |
| Level curve | Set of input points producing the same scalar output |


## Section 1 guided worksheet

1. State whether each mapping is scalar-valued or vector-valued:

   $$
   f(x,y)=x^2+y^2,
   $$

   $$
   F(x,y)=
   \begin{bmatrix}
   x+y\\
   x-y
   \end{bmatrix}.
   $$

2. Evaluate

   $$
   g(x,y,z)=xy+yz+xz
   $$

   at $$(1,2,3).$$

3. Describe the domain of

   $$
   h(x,y)=\ln(4-x^2-y^2).
   $$

4. Explain how a level curve differs from a training trajectory drawn on the same contour plot.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
1. The first function produces one number, so it is scalar-valued. The second produces two coordinates, so it is vector-valued.

2. Substitute:

   $$
   g(1,2,3)=(1)(2)+(2)(3)+(1)(3)
   =2+6+3
   =\boxed{11}.
   $$

3. A logarithm requires a positive argument:

   $$
   4-x^2-y^2>0.
   $$

   Thus

   $$
   x^2+y^2<4.
   $$

   The domain is the open disk of radius two centered at the origin. The boundary is excluded.

4. A level curve is the full set of points with one fixed loss value. A training trajectory is an ordered sequence of parameter points produced by an algorithm and can cross many level curves.
""")


---

# Section 2: Partial Derivatives

## 1. Changing one input while freezing the others

For a function $$f(x,y),$$ the **partial derivative with respect to** $$x$$ measures how the output changes when $$x$$ changes while $$y$$ is held fixed:

$$
\frac{\partial f}{\partial x}(x,y)
=\lim_{h\to 0}\frac{f(x+h,y)-f(x,y)}{h}.
$$

Similarly,

$$
\frac{\partial f}{\partial y}(x,y)
=\lim_{h\to 0}\frac{f(x,y+h)-f(x,y)}{h}.
$$

Common notations are

$$
f_x,
\qquad
f_y,
\qquad
\partial_x f,
\qquad
\partial_y f.
$$

The rounded symbol distinguishes partial differentiation from ordinary one-variable differentiation.


## 2. Fully worked partial derivatives

Let

$$
f(x,y)=x^2y+3xy^2.
$$

To compute $$f_x,$$ treat $$y$$ as a constant:

$$
\frac{\partial}{\partial x}(x^2y)=y\frac{\partial}{\partial x}(x^2)=2xy,
$$

and

$$
\frac{\partial}{\partial x}(3xy^2)=3y^2\frac{\partial}{\partial x}(x)=3y^2.
$$

Therefore

$$
\boxed{f_x(x,y)=2xy+3y^2.}
$$

To compute $$f_y,$$ treat $$x$$ as a constant:

$$
\frac{\partial}{\partial y}(x^2y)=x^2,
$$

and

$$
\frac{\partial}{\partial y}(3xy^2)=3x(2y)=6xy.
$$

Therefore

$$
\boxed{f_y(x,y)=x^2+6xy.}
$$

At $$(1,2),$$

$$
f_x(1,2)=2(1)(2)+3(2^2)=4+12=16,
$$

and

$$
f_y(1,2)=1^2+6(1)(2)=1+12=13.
$$


## 3. Partial derivatives as cross-sectional slopes

On the surface $$z=f(x,y),$$ fixing $$y=y_0$$ produces a one-variable cross section

$$
z=f(x,y_0).
$$

Its slope at $$x=x_0$$ is $$f_x(x_0,y_0).$$ Fixing $$x=x_0$$ instead produces a different cross section whose slope is $$f_y(x_0,y_0).$$

The same surface can therefore have different slopes at one point depending on the direction of motion.


In [ ]:
x_cross = np.linspace(-2, 2, 250)
y_cross = np.linspace(-2, 2, 250)
X_cross, Y_cross = np.meshgrid(x_cross, y_cross)
Z_cross = X_cross**2 + 2*Y_cross**2

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(X_cross, Y_cross, Z_cross, cmap="Blues", alpha=0.55)
ax1.plot(x_cross, np.ones_like(x_cross), x_cross**2+2,
         color="crimson", linewidth=3, label="hold y = 1")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("f")
ax1.set_title("Partial derivative in the x direction")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.plot_surface(X_cross, Y_cross, Z_cross, cmap="Greens", alpha=0.55)
ax2.plot(np.ones_like(y_cross), y_cross, 1+2*y_cross**2,
         color="purple", linewidth=3, label="hold x = 1")
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_zlabel("f")
ax2.set_title("Partial derivative in the y direction")
plt.tight_layout()
plt.show()


## 4. Second and mixed partial derivatives

Differentiate a partial derivative again:

$$
f_{xx}=\frac{\partial}{\partial x}(f_x),
\qquad
f_{yy}=\frac{\partial}{\partial y}(f_y).
$$

Mixed partial derivatives differentiate with respect to different variables:

$$
f_{xy}=\frac{\partial}{\partial y}(f_x),
\qquad
f_{yx}=\frac{\partial}{\partial x}(f_y).
$$

For

$$
f(x,y)=x^2y+3xy^2,
$$

we found

$$
f_x=2xy+3y^2.
$$

Thus

$$
f_{xy}=2x+6y.
$$

Also,

$$
f_y=x^2+6xy,
$$

so

$$
f_{yx}=2x+6y.
$$

When the mixed partials are continuous near a point, Clairaut's theorem gives

$$
f_{xy}=f_{yx}.
$$


## 5. Direct AI example: one observation and two weights

Let

$$
\widehat{y}=w_1x_1+w_2x_2
$$

and squared loss

$$
L(w_1,w_2)=(w_1x_1+w_2x_2-y)^2.
$$

Define the residual

$$
r=w_1x_1+w_2x_2-y.
$$

Then

$$
L=r^2.
$$

Holding $$w_2$$ fixed, the chain rule gives

$$
\frac{\partial L}{\partial w_1}
=2r\frac{\partial r}{\partial w_1}
=2rx_1.
$$

Similarly,

$$
\frac{\partial L}{\partial w_2}
=2r\frac{\partial r}{\partial w_2}
=2rx_2.
$$

Both derivatives contain the same residual but are scaled by different features. This tells the optimizer how each weight affects the same prediction error.


> **Common mistakes in Section 2**
>
> - Differentiating every variable when taking one partial derivative.
> - Treating a frozen variable as zero instead of as a constant.
> - Losing the distinction between $$f_{xy}$$ and a product $$f_xf_y.$$
> - Assuming mixed partial derivatives always agree without regularity conditions.
> - Confusing a partial derivative with the total change when several inputs vary together.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Partial derivative | Sensitivity to one coordinate while others are fixed |
| Cross section | One-variable slice of a multivariable graph |
| First partial | Partial derivative taken once |
| Second partial | Partial derivative taken twice |
| Mixed partial | Successive differentiation with respect to different variables |
| Residual | Prediction minus target |
| Feature sensitivity | Change in loss associated with a weight-feature pathway |


## Section 2 guided worksheet

Let

$$
f(x,y)=x^3y^2+4xy-5y.
$$

1. Compute $$f_x$$ and $$f_y.$$
2. Evaluate both at $$(1,2).$$
3. Compute $$f_{xy}$$ and $$f_{yx}.$$
4. Explain in words what $$f_x(1,2)$$ measures.


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
1. Holding $$y$$ fixed,

   $$
   f_x=3x^2y^2+4y.
   $$

   Holding $$x$$ fixed,

   $$
   f_y=2x^3y+4x-5.
   $$

2. At $$(1,2),$$

   $$
   f_x(1,2)=3(1)^2(2)^2+4(2)=12+8=\boxed{20},
   $$

   and

   $$
   f_y(1,2)=2(1)^3(2)+4(1)-5=4+4-5=\boxed{3}.
   $$

3. Differentiate:

   $$
   f_{xy}=6x^2y+4
   $$

   and

   $$
   f_{yx}=6x^2y+4.
   $$

4. It is the instantaneous rate of output change when $$x$$ varies near one while $$y$$ is held fixed at two.
""")


---

# Section 3: Gradient Vectors and Directional Derivatives

## 1. Collecting partial derivatives

For a scalar function $$f:\mathbb{R}^n\to\mathbb{R},$$ the **gradient** is the vector of first partial derivatives:

$$
\boxed{\nabla f(\mathbf{x})=
\begin{bmatrix}
\dfrac{\partial f}{\partial x_1}\\
\dfrac{\partial f}{\partial x_2}\\
\vdots\\
\dfrac{\partial f}{\partial x_n}
\end{bmatrix}.}
$$

For two variables,

$$
\nabla f(x,y)=
\begin{bmatrix}
f_x(x,y)\\
f_y(x,y)
\end{bmatrix}.
$$

The gradient lives in the input space. Each component reports sensitivity to one coordinate.


## 2. Directional derivative

Let

$$
\mathbf{u}=
\begin{bmatrix}
u_1\\
u_2
\end{bmatrix}
$$

be a **unit vector**, so

$$
\lVert\mathbf{u}\rVert_2=1.
$$

The directional derivative of $$f$$ at point $$\mathbf{p}$$ in direction $$\mathbf{u}$$ is

$$
\boxed{D_{\mathbf{u}}f(\mathbf{p})
=\nabla f(\mathbf{p})\cdot\mathbf{u}.}
$$

It measures the instantaneous rate of change per unit distance traveled in that direction. A non-unit direction would mix direction with travel speed.


## 3. Derive the directional-derivative formula

Move along the line

$$
\mathbf{r}(t)=\mathbf{p}+t\mathbf{u}.
$$

For two variables,

$$
x(t)=p_1+tu_1,
\qquad
y(t)=p_2+tu_2.
$$

Apply the multivariable chain rule to $$f(x(t),y(t)):$$

$$
\frac{d}{dt}f(x(t),y(t))
=f_x\frac{dx}{dt}+f_y\frac{dy}{dt}.
$$

Since

$$
\frac{dx}{dt}=u_1
\qquad\text{and}\qquad
\frac{dy}{dt}=u_2,
$$

evaluating at $$t=0$$ gives

$$
D_{\mathbf{u}}f(\mathbf{p})
=f_x(\mathbf{p})u_1+f_y(\mathbf{p})u_2
=\nabla f(\mathbf{p})\cdot\mathbf{u}.
$$


## 4. Detailed directional example

Let

$$
f(x,y)=x^2+2y^2
$$

and evaluate at

$$
\mathbf{p}=(1,1).
$$

The partial derivatives are

$$
f_x=2x
$$

and

$$
f_y=4y.
$$

Thus

$$
\nabla f(1,1)=
\begin{bmatrix}
2\\
4
\end{bmatrix}.
$$

Use the unit direction

$$
\mathbf{u}=
\begin{bmatrix}
3/5\\
4/5
\end{bmatrix}.
$$

Then

$$
D_{\mathbf{u}}f(1,1)
=
\begin{bmatrix}
2\\
4
\end{bmatrix}
\cdot
\begin{bmatrix}
3/5\\
4/5
\end{bmatrix}.
$$

Compute:

$$
D_{\mathbf{u}}f(1,1)
=\frac{6}{5}+\frac{16}{5}
=\boxed{\frac{22}{5}=4.4}.
$$


## 5. Why the gradient is steepest ascent

For any unit vector $$\mathbf{u},$$ the Cauchy-Schwarz inequality gives

$$
\nabla f\cdot\mathbf{u}
\leq \lVert\nabla f\rVert_2\lVert\mathbf{u}\rVert_2.
$$

Because $$\lVert\mathbf{u}\rVert_2=1,$$

$$
D_{\mathbf{u}}f
\leq \lVert\nabla f\rVert_2.
$$

Equality occurs when $$\mathbf{u}$$ points in the same direction as $$\nabla f.$$ Therefore:

- $$\nabla f$$ points toward steepest local increase;
- $$-\nabla f$$ points toward steepest local decrease; and
- $$\lVert\nabla f\rVert_2$$ is the maximum directional rate of increase.

At points where $$\nabla f=\mathbf{0},$$ no direction has a nonzero first-order change.


In [ ]:
x_grad = np.linspace(-2.2, 2.2, 25)
y_grad = np.linspace(-2.2, 2.2, 25)
X_grad, Y_grad = np.meshgrid(x_grad, y_grad)
F_grad = X_grad**2 + 2*Y_grad**2
GX_grad = 2*X_grad
GY_grad = 4*Y_grad

fig, ax = plt.subplots(figsize=(7, 6))
contours_grad = ax.contour(X_grad, Y_grad, F_grad, levels=14, cmap="viridis")
ax.quiver(X_grad[::2, ::2], Y_grad[::2, ::2],
          GX_grad[::2, ::2], GY_grad[::2, ::2], color="crimson", alpha=0.75)
ax.clabel(contours_grad, inline=True, fontsize=7)
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Gradients cross level curves in the steepest direction")
plt.show()


## 6. Tangent plane and linearization

For $$z=f(x,y),$$ the tangent plane at $$(a,b)$$ is

$$
z=f(a,b)+f_x(a,b)(x-a)+f_y(a,b)(y-b).
$$

This gives the local linear approximation

$$
f(a+\Delta x,b+\Delta y)
\approx f(a,b)+\nabla f(a,b)\cdot
\begin{bmatrix}
\Delta x\\
\Delta y
\end{bmatrix}.
$$

For

$$
f(x,y)=x^2+y^2
$$

at $$(1,2),$$

$$
f(1,2)=5,
\qquad
f_x(1,2)=2,
\qquad
f_y(1,2)=4.
$$

Thus

$$
z=5+2(x-1)+4(y-2).
$$

Simplifying,

$$
\boxed{z=2x+4y-5.}
$$


> **Common mistakes in Section 3**
>
> - Forgetting to normalize a direction vector.
> - Treating the gradient as a scalar.
> - Assuming the gradient points downhill rather than uphill.
> - Confusing gradient magnitude with a selected directional derivative.
> - Believing the tangent plane is globally exact for a nonlinear surface.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Gradient | Vector of first partial derivatives |
| Directional derivative | Rate of change in a specified unit direction |
| Unit direction | Direction vector with Euclidean norm one |
| Steepest ascent | Unit direction maximizing the directional derivative |
| Steepest descent | Opposite of the gradient direction |
| Tangent plane | Best first-order plane approximation at a point |
| Linearization | Local approximation using the function value and gradient |


## Section 3 guided worksheet

Let

$$
f(x,y)=3x^2+xy+y^2.
$$

1. Find $$\nabla f(x,y).$$
2. Evaluate the gradient at $$(1,-1).$$
3. Normalize the direction vector

   $$
   \mathbf{v}=
   \begin{bmatrix}
   1\\
   2
   \end{bmatrix}.
   $$

4. Compute the directional derivative in that unit direction.
5. Give the unit direction of steepest decrease.


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
1. The partial derivatives are

   $$
   f_x=6x+y
   $$

   and

   $$
   f_y=x+2y.
   $$

   Therefore

   $$
   \nabla f(x,y)=
   \begin{bmatrix}
   6x+y\\
   x+2y
   \end{bmatrix}.
   $$

2. At $$(1,-1),$$

   $$
   \nabla f(1,-1)=
   \begin{bmatrix}
   6-1\\
   1-2
   \end{bmatrix}
   =
   \begin{bmatrix}
   5\\
   -1
   \end{bmatrix}.
   $$

3. The norm is

   $$
   \lVert\mathbf{v}\rVert_2=\sqrt{1^2+2^2}=\sqrt{5}.
   $$

   Hence

   $$
   \mathbf{u}=\frac{1}{\sqrt{5}}
   \begin{bmatrix}
   1\\
   2
   \end{bmatrix}.
   $$

4. The directional derivative is

   $$
   D_{\mathbf{u}}f(1,-1)
   =
   \begin{bmatrix}
   5\\
   -1
   \end{bmatrix}
   \cdot
   \frac{1}{\sqrt{5}}
   \begin{bmatrix}
   1\\
   2
   \end{bmatrix}
   =\frac{5-2}{\sqrt{5}}
   =\boxed{\frac{3}{\sqrt{5}}}.
   $$

5. The gradient norm is $$\sqrt{26},$$ so the steepest-descent unit direction is

   $$
   \boxed{-\frac{1}{\sqrt{26}}
   \begin{bmatrix}
   5\\
   -1
   \end{bmatrix}
   =
   \frac{1}{\sqrt{26}}
   \begin{bmatrix}
   -5\\
   1
   \end{bmatrix}.}
   $$
""")


---

# Section 4: Multivariable Chain Rule, Computation Graphs, and Jacobians

## 1. Several changing inputs

Suppose

$$
z=f(x,y),
$$

while both inputs depend on $$t$$:

$$
x=x(t),
\qquad
y=y(t).
$$

Then the total derivative is

$$
\boxed{\frac{dz}{dt}
=\frac{\partial f}{\partial x}\frac{dx}{dt}
+\frac{\partial f}{\partial y}\frac{dy}{dt}.}
$$

Each path from $$t$$ to $$z$$ contributes a product of local derivatives, and contributions from different paths are added.


## 2. Detailed chain-rule example

Let

$$
z=x^2y+e^y,
$$

$$
x=t^2,
$$

and

$$
y=3t-1.
$$

First compute local partial derivatives:

$$
z_x=2xy,
$$

$$
z_y=x^2+e^y.
$$

Next compute the input derivatives:

$$
\frac{dx}{dt}=2t,
$$

$$
\frac{dy}{dt}=3.
$$

Apply the multivariable chain rule:

$$
\frac{dz}{dt}
=(2xy)(2t)+(x^2+e^y)(3).
$$

Replace $$x=t^2$$ and $$y=3t-1$$:

$$
\boxed{\frac{dz}{dt}
=4t^3(3t-1)+3t^4+3e^{3t-1}.}
$$


## 3. Branching paths in backpropagation

Suppose a parameter $$w$$ affects two intermediate quantities:

$$
u=u(w),
\qquad
v=v(w),
$$

and the loss is $$L(u,v).$$ Then

$$
\boxed{\frac{dL}{dw}
=\frac{\partial L}{\partial u}\frac{du}{dw}
+\frac{\partial L}{\partial v}\frac{dv}{dw}.}
$$

The derivative contribution along each path is a product. Contributions from separate paths are summed because the parameter influences the loss in more than one way.

This path-product and path-sum rule is the essence of backpropagation on a directed acyclic computation graph.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off")
positions_graph = {
    "w": (0.08, 0.5), "u": (0.38, 0.72), "v": (0.38, 0.28), "L": (0.82, 0.5)
}
for label, position in positions_graph.items():
    ax.text(position[0], position[1], label, ha="center", va="center", fontsize=13,
            bbox=dict(boxstyle="round,pad=0.55", facecolor="#dbeafe", edgecolor="#1d4ed8"))
for start, end in [("w", "u"), ("w", "v"), ("u", "L"), ("v", "L")]:
    ax.annotate("", xy=positions_graph[end], xytext=positions_graph[start],
                arrowprops=dict(arrowstyle="->", linewidth=2, shrinkA=24, shrinkB=24))
ax.set_title("Multiply derivatives along paths; add contributions across paths")
plt.show()


## 4. Fully worked two-feature neuron

Consider

$$
z=w_1x_1+w_2x_2+b,
$$

$$
\widehat{y}=\sigma(z)=\frac{1}{1+e^{-z}},
$$

and

$$
L=\frac{1}{2}(\widehat{y}-y)^2.
$$

Use

$$
\mathbf{x}=
\begin{bmatrix}
2\\
-1
\end{bmatrix},
\qquad
\mathbf{w}=
\begin{bmatrix}
0.5\\
-0.25
\end{bmatrix},
\qquad
b=0.1,
\qquad
y=1.
$$

### Forward pass

$$
z=(0.5)(2)+(-0.25)(-1)+0.1
=1+0.25+0.1
=1.35.
$$

$$
\widehat{y}=\sigma(1.35)\approx 0.794130.
$$

$$
L=\frac{1}{2}(0.794130-1)^2
\approx 0.021191.
$$

### Backward pass

$$
\frac{\partial L}{\partial \widehat{y}}=\widehat{y}-y,
$$

$$
\frac{\partial \widehat{y}}{\partial z}
=\widehat{y}(1-\widehat{y}),
$$

and

$$
\frac{\partial z}{\partial w_i}=x_i.
$$

Thus

$$
\frac{\partial L}{\partial w_i}
=(\widehat{y}-y)\widehat{y}(1-\widehat{y})x_i.
$$

The bias gradient uses

$$
\frac{\partial z}{\partial b}=1.
$$


In [ ]:
x_neuron8 = np.array([2.0, -1.0])
w_neuron8 = np.array([0.5, -0.25])
b_neuron8 = 0.1
y_neuron8 = 1.0
z_neuron8 = w_neuron8 @ x_neuron8 + b_neuron8
yhat_neuron8 = 1/(1+np.exp(-z_neuron8))
loss_neuron8 = 0.5*(yhat_neuron8-y_neuron8)**2
common_neuron8 = (yhat_neuron8-y_neuron8)*yhat_neuron8*(1-yhat_neuron8)
grad_w_neuron8 = common_neuron8*x_neuron8
grad_b_neuron8 = common_neuron8

print("z =", z_neuron8)
print("prediction =", yhat_neuron8)
print("loss =", loss_neuron8)
print("gradient with respect to weights =", grad_w_neuron8)
print("gradient with respect to bias =", grad_b_neuron8)


## 5. Jacobian matrices

For a vector-valued function

$$
F:\mathbb{R}^n\to\mathbb{R}^m,
$$

the **Jacobian matrix** collects every first partial derivative:

$$
J_F(\mathbf{x})=
\begin{bmatrix}
\dfrac{\partial F_1}{\partial x_1}&\cdots&\dfrac{\partial F_1}{\partial x_n}\\
\vdots&\ddots&\vdots\\
\dfrac{\partial F_m}{\partial x_1}&\cdots&\dfrac{\partial F_m}{\partial x_n}
\end{bmatrix}.
$$

Its shape is

$$
m\times n,
$$

matching output coordinates by input coordinates.

For a scalar-valued function, the same first derivatives form the gradient. The gradient is commonly written as a column vector, whereas the derivative map may be represented as a row; conventions should be stated clearly.


> **Common mistakes in Section 4**
>
> - Multiplying contributions from separate branches instead of adding them.
> - Adding local derivatives along a single path instead of multiplying them.
> - Forgetting which quantities are held fixed in a partial derivative.
> - Reversing Jacobian dimensions.
> - Treating a computation graph as merely a picture rather than a dependency structure.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Total derivative | Change when all dependent inputs vary together |
| Multivariable chain rule | Sum of products across dependency paths |
| Branch | One dependency path in a computation graph |
| Backpropagation | Reverse accumulation of derivatives from loss to parameters |
| Jacobian | Matrix of first derivatives for a vector-valued function |
| Local derivative | Sensitivity of one node to an immediate predecessor |
| Directed acyclic graph | Dependency graph with directed edges and no directed cycle |


## Section 4 guided worksheet

Let

$$
L(u,v)=u^2+uv,
$$

$$
u(w)=w^2,
$$

and

$$
v(w)=3w+1.
$$

1. Compute $$L_u$$ and $$L_v.$$
2. Compute $$du/dw$$ and $$dv/dw.$$
3. Use the multivariable chain rule to find $$dL/dw.$$
4. Substitute directly first and verify the same derivative.


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. We have

   $$
   L_u=2u+v
   $$

   and

   $$
   L_v=u.
   $$

2. The local derivatives are

   $$
   \frac{du}{dw}=2w
   $$

   and

   $$
   \frac{dv}{dw}=3.
   $$

3. Apply the chain rule:

   $$
   \frac{dL}{dw}
   =(2u+v)(2w)+u(3).
   $$

   Substitute $$u=w^2$$ and $$v=3w+1$$:

   $$
   \frac{dL}{dw}
   =\bigl(2w^2+3w+1\bigr)(2w)+3w^2.
   $$

   Expand:

   $$
   \boxed{\frac{dL}{dw}=4w^3+9w^2+2w.}
   $$

4. Direct substitution gives

   $$
   L(w)=w^4+w^2(3w+1)
   =w^4+3w^3+w^2.
   $$

   Differentiating gives

   $$
   L'(w)=4w^3+9w^2+2w,
   $$

   confirming the result.
""")


---

# Section 5: Critical Points, Hessians, and Local Shape

## 1. Multivariable critical points

For a differentiable scalar function, an interior **critical point** satisfies

$$
\nabla f(\mathbf{x}^*)=\mathbf{0}.
$$

This condition says every first-order coordinate sensitivity is zero. As in one variable, it identifies candidates rather than guaranteeing minima.

A point can be:

- a local minimum;
- a local maximum;
- a saddle point; or
- a degenerate point requiring higher-order analysis.


## 2. The Hessian matrix

The **Hessian** collects second partial derivatives of a scalar function:

$$
H_f(x,y)=
\begin{bmatrix}
f_{xx}&f_{xy}\\
f_{yx}&f_{yy}
\end{bmatrix}.
$$

It describes local curvature in different directions. When mixed partials are continuous, the Hessian is symmetric.

For a two-variable critical point, define

$$
D=f_{xx}f_{yy}-(f_{xy})^2.
$$

Then:

- if $$D>0$$ and $$f_{xx}>0,$$ there is a local minimum;
- if $$D>0$$ and $$f_{xx}<0,$$ there is a local maximum;
- if $$D<0,$$ there is a saddle point; and
- if $$D=0,$$ the test is inconclusive.


## 3. Detailed positive-curvature example

Let

$$
f(x,y)=x^2+4xy+5y^2.
$$

The gradient is

$$
\nabla f=
\begin{bmatrix}
2x+4y\\
4x+10y
\end{bmatrix}.
$$

Set both components equal to zero:

$$
2x+4y=0,
$$

$$
4x+10y=0.
$$

The first equation gives

$$
x=-2y.
$$

Substitute into the second:

$$
4(-2y)+10y=0,
$$

$$
-8y+10y=0,
$$

$$
2y=0.
$$

Thus $$y=0$$ and $$x=0.$$

The Hessian is

$$
H_f=
\begin{bmatrix}
2&4\\
4&10
\end{bmatrix}.
$$

Compute

$$
D=(2)(10)-4^2=20-16=4>0.
$$

Since $$f_{xx}=2>0,$$ the origin is a strict local minimum. Because this quadratic curves upward in every nonzero direction, it is also the global minimum.


## 4. Saddle point example

Consider

$$
g(x,y)=x^2-y^2.
$$

Its gradient is

$$
\nabla g=
\begin{bmatrix}
2x\\
-2y
\end{bmatrix},
$$

so the origin is critical. The Hessian is

$$
H_g=
\begin{bmatrix}
2&0\\
0&-2
\end{bmatrix}.
$$

Its determinant is

$$
D=(2)(-2)-0=-4<0.
$$

Therefore the origin is a saddle point. Along the $$x$$ axis,

$$
g(x,0)=x^2\geq 0,
$$

but along the $$y$$ axis,

$$
g(0,y)=-y^2\leq 0.
$$

The point is uphill in one direction and downhill in another.


In [ ]:
x_hessian = np.linspace(-2, 2, 180)
y_hessian = np.linspace(-2, 2, 180)
X_hessian, Y_hessian = np.meshgrid(x_hessian, y_hessian)
minimum_surface = X_hessian**2 + 4*X_hessian*Y_hessian + 5*Y_hessian**2
saddle_surface = X_hessian**2 - Y_hessian**2

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(X_hessian, Y_hessian, minimum_surface, cmap="viridis")
ax1.set_title("Positive curvature: minimum")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.plot_surface(X_hessian, Y_hessian, saddle_surface, cmap="coolwarm")
ax2.set_title("Mixed curvature: saddle")
ax2.set_xlabel("x")
ax2.set_ylabel("y")
plt.tight_layout()
plt.show()


## 5. Convexity

A function is **convex** on a convex domain if every line segment between two graph points lies on or above the graph. Algebraically,

$$
f(t\mathbf{x}+(1-t)\mathbf{y})
\leq tf(\mathbf{x})+(1-t)f(\mathbf{y}),
\qquad 0\leq t\leq 1.
$$

For a differentiable convex function, every local minimum is global. For a twice-differentiable function, a positive semidefinite Hessian throughout a convex domain is a standard convexity test.

Many simple regression losses are convex, but neural-network training losses are generally nonconvex. Nonconvex landscapes can contain saddle points, flat regions, and multiple minima.


> **Common mistakes in Section 5**
>
> - Declaring every zero-gradient point a minimum.
> - Using only one second partial derivative to classify a point.
> - Confusing a saddle point with a local maximum.
> - Applying the two-variable determinant test when $$D=0$$ and forcing a conclusion.
> - Assuming every AI loss is convex.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Critical point | Interior point with zero gradient or missing derivatives |
| Hessian | Matrix of second partial derivatives |
| Saddle point | Point with increasing and decreasing directions |
| Positive definite | Producing positive quadratic curvature in every nonzero direction |
| Positive semidefinite | Producing nonnegative quadratic curvature in every direction |
| Convex function | Function lying below chords between graph points |
| Nonconvex landscape | Objective surface not satisfying global convexity |


## Section 5 guided worksheet

Classify the critical point of

$$
f(x,y)=3x^2-2xy+2y^2.
$$

1. Compute the gradient.
2. Solve for the critical point.
3. Construct the Hessian.
4. Apply the determinant test.
5. Explain whether the conclusion is local or global and why.


In [ ]:
reveal_markdown("Open Section 5 solutions", r"""
1. The gradient is

   $$
   \nabla f=
   \begin{bmatrix}
   6x-2y\\
   -2x+4y
   \end{bmatrix}.
   $$

2. Solve

   $$
   6x-2y=0
   $$

   and

   $$
   -2x+4y=0.
   $$

   The first gives $$y=3x.$$ Substitute into the second:

   $$
   -2x+4(3x)=10x=0.
   $$

   Thus $$x=0$$ and $$y=0.$$

3. The Hessian is

   $$
   H_f=
   \begin{bmatrix}
   6&-2\\
   -2&4
   \end{bmatrix}.
   $$

4. Compute

   $$
   D=(6)(4)-(-2)^2=24-4=20>0.
   $$

   Since $$f_{xx}=6>0,$$ the origin is a strict local minimum.

5. The function is a quadratic with a positive-definite Hessian everywhere, so it is strictly convex. Therefore the local minimum is also the unique global minimum.
""")


---

# Section 6: Gradient Descent with Several Parameters

## 1. Vector update rule

For parameter vector $$\boldsymbol{\theta},$$ multivariable gradient descent is

$$
\boxed{\boldsymbol{\theta}_{k+1}
=\boldsymbol{\theta}_k-\eta\nabla L(\boldsymbol{\theta}_k).}
$$

For two parameters,

$$
\begin{bmatrix}
w_{1,k+1}\\
w_{2,k+1}
\end{bmatrix}
=
\begin{bmatrix}
w_{1,k}\\
w_{2,k}
\end{bmatrix}
-\eta
\begin{bmatrix}
\dfrac{\partial L}{\partial w_1}\\
\dfrac{\partial L}{\partial w_2}
\end{bmatrix}_{\boldsymbol{\theta}_k}.
$$

All components are updated simultaneously from the gradient evaluated at the current point.


## 2. Fully worked two-parameter updates

Let

$$
L(w_1,w_2)=(w_1-2)^2+2(w_2+1)^2.
$$

The gradient is

$$
\nabla L=
\begin{bmatrix}
2(w_1-2)\\
4(w_2+1)
\end{bmatrix}.
$$

Start with

$$
\boldsymbol{\theta}_0=
\begin{bmatrix}
0\\
2
\end{bmatrix}
$$

and choose $$\eta=0.1.$$

At the initial point,

$$
\nabla L(0,2)=
\begin{bmatrix}
2(0-2)\\
4(2+1)
\end{bmatrix}
=
\begin{bmatrix}
-4\\
12
\end{bmatrix}.
$$

Thus

$$
\boldsymbol{\theta}_1
=
\begin{bmatrix}
0\\
2
\end{bmatrix}
-0.1
\begin{bmatrix}
-4\\
12
\end{bmatrix}
=
\begin{bmatrix}
0.4\\
0.8
\end{bmatrix}.
$$

At this new point,

$$
\nabla L(0.4,0.8)=
\begin{bmatrix}
2(0.4-2)\\
4(0.8+1)
\end{bmatrix}
=
\begin{bmatrix}
-3.2\\
7.2
\end{bmatrix}.
$$

Therefore

$$
\boldsymbol{\theta}_2
=
\begin{bmatrix}
0.4\\
0.8
\end{bmatrix}
-0.1
\begin{bmatrix}
-3.2\\
7.2
\end{bmatrix}
=
\boxed{
\begin{bmatrix}
0.72\\
0.08
\end{bmatrix}}.
$$


In [ ]:
def loss_descent8(theta):
    w1, w2 = theta
    return (w1-2)**2 + 2*(w2+1)**2

def gradient_descent8(theta):
    w1, w2 = theta
    return np.array([2*(w1-2), 4*(w2+1)])

theta_path8 = [np.array([0.0, 2.0])]
eta_descent8 = 0.1
for _ in range(18):
    current = theta_path8[-1]
    theta_path8.append(current-eta_descent8*gradient_descent8(current))
theta_path8 = np.array(theta_path8)

w1_grid8 = np.linspace(-0.5, 3.5, 250)
w2_grid8 = np.linspace(-2.8, 2.8, 250)
W1_grid8, W2_grid8 = np.meshgrid(w1_grid8, w2_grid8)
L_grid8 = (W1_grid8-2)**2+2*(W2_grid8+1)**2

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(W1_grid8, W2_grid8, L_grid8, levels=18, cmap="viridis")
ax.plot(theta_path8[:, 0], theta_path8[:, 1], color="crimson", marker="o",
        markersize=4, label="gradient-descent path")
ax.scatter([2], [-1], color="black", marker="*", s=150, label="minimum")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_title("A vector update crosses loss contours")
ax.legend()
plt.show()


## 3. Scale and conditioning

If one direction has much larger curvature than another, one learning rate must respect the steep direction while progress can be slow in the flat direction. This creates narrow, elongated contours and a zigzagging path.

**Feature scaling** transforms input features to comparable numerical scales. It can improve the geometry of simple optimization problems because weights no longer compensate for drastically different units.

**Conditioning** describes how differently a function responds across directions. Poorly conditioned quadratic losses have a large ratio between their largest and smallest curvature scales.

Scaling does not change every optimization issue, but it is an important practical preprocessing idea.


## 4. Millions of parameters use the same principle

For

$$
\boldsymbol{\theta}\in\mathbb{R}^p,
$$

the gradient still has one component per parameter:

$$
\nabla L(\boldsymbol{\theta})
\in\mathbb{R}^p.
$$

Backpropagation computes these components efficiently by reusing intermediate derivatives rather than differentiating the entire model separately for every parameter.

The geometry becomes impossible to draw when $$p$$ is large, but the vector update remains

$$
\boldsymbol{\theta}_{k+1}
=\boldsymbol{\theta}_k-\eta\nabla L(\boldsymbol{\theta}_k).
$$

Visualization is therefore an analogy for the algebra, not a claim that real training occurs on a two-dimensional surface.


> **Common mistakes in Section 6**
>
> - Updating one component before computing the others from the old point.
> - Mixing gradient dimensions with parameter dimensions.
> - Assuming the same learning rate is equally safe in all loss landscapes.
> - Interpreting a two-dimensional contour plot as the literal shape of a high-dimensional network loss.
> - Assuming feature scaling guarantees convergence.

## Section 6 terminology

| Term | Meaning |
|---|---|
| Vector gradient descent | Simultaneous update opposite the multivariable gradient |
| Parameter vector | Ordered collection of trainable quantities |
| Feature scaling | Transformation placing feature magnitudes on comparable scales |
| Conditioning | Sensitivity imbalance across different directions |
| Loss landscape | Objective viewed over parameter space |
| Training path | Sequence of parameter vectors generated by updates |
| High-dimensional optimization | Optimization involving many parameter coordinates |


## Section 6 guided worksheet

Use

$$
L(u,v)=3(u-1)^2+(v+2)^2
$$

with initial point

$$
(u_0,v_0)=(0,0)
$$

and learning rate $$\eta=0.1.$$

1. Compute the gradient.
2. Compute the first two vector updates.
3. Identify the exact minimizer.
4. Explain which coordinate initially changes more and why.


In [ ]:
reveal_markdown("Open Section 6 solutions", r"""
1. The gradient is

   $$
   \nabla L(u,v)=
   \begin{bmatrix}
   6(u-1)\\
   2(v+2)
   \end{bmatrix}.
   $$

2. At $$(0,0),$$

   $$
   \nabla L(0,0)=
   \begin{bmatrix}
   -6\\
   4
   \end{bmatrix}.
   $$

   Therefore

   $$
   \begin{bmatrix}
   u_1\\
   v_1
   \end{bmatrix}
   =
   \begin{bmatrix}
   0\\
   0
   \end{bmatrix}
   -0.1
   \begin{bmatrix}
   -6\\
   4
   \end{bmatrix}
   =
   \begin{bmatrix}
   0.6\\
   -0.4
   \end{bmatrix}.
   $$

   At the new point,

   $$
   \nabla L(0.6,-0.4)=
   \begin{bmatrix}
   6(-0.4)\\
   2(1.6)
   \end{bmatrix}
   =
   \begin{bmatrix}
   -2.4\\
   3.2
   \end{bmatrix}.
   $$

   Hence

   $$
   \begin{bmatrix}
   u_2\\
   v_2
   \end{bmatrix}
   =
   \begin{bmatrix}
   0.6\\
   -0.4
   \end{bmatrix}
   -0.1
   \begin{bmatrix}
   -2.4\\
   3.2
   \end{bmatrix}
   =
   \boxed{
   \begin{bmatrix}
   0.84\\
   -0.72
   \end{bmatrix}}.
   $$

3. The exact minimizer is $$(1,-2).$$

4. Initially, the magnitude of the $$u$$ derivative is six while the magnitude of the $$v$$ derivative is four, so the same learning rate produces a larger first change in $$u.$$
""")


---

# Integrated Instructor Lab: Two-Parameter Linear Regression

Use the data

$$
(x_1,y_1)=(0,1),
\qquad
(x_2,y_2)=(1,3),
\qquad
(x_3,y_3)=(2,5)
$$

and model

$$
\widehat{y}=wx+b.
$$

The mean squared error is

$$
L(w,b)=\frac{1}{3}
\left[(b-1)^2+(w+b-3)^2+(2w+b-5)^2\right].
$$

### Partial derivative with respect to the slope

$$
L_w=\frac{2}{3}
\left[0(b-1)+1(w+b-3)+2(2w+b-5)\right].
$$

Simplify inside the brackets:

$$
(w+b-3)+(4w+2b-10)=5w+3b-13.
$$

Thus

$$
\boxed{L_w=\frac{2}{3}(5w+3b-13).}
$$

### Partial derivative with respect to the intercept

$$
L_b=\frac{2}{3}
\left[(b-1)+(w+b-3)+(2w+b-5)\right].
$$

Simplify:

$$
(b-1)+(w+b-3)+(2w+b-5)=3w+3b-9.
$$

Therefore

$$
\boxed{L_b=\frac{2}{3}(3w+3b-9)=2(w+b-3).}
$$


## Solve for the exact optimum

Set both partial derivatives to zero:

$$
5w+3b-13=0,
$$

$$
w+b-3=0.
$$

The second equation gives

$$
b=3-w.
$$

Substitute into the first:

$$
5w+3(3-w)-13=0.
$$

Expand:

$$
5w+9-3w-13=0.
$$

Combine terms:

$$
2w-4=0.
$$

Thus

$$
w=2
$$

and

$$
b=3-2=1.
$$

Therefore

$$
\boxed{(w^*,b^*)=(2,1).}
$$

The fitted line is $$\widehat{y}=2x+1,$$ which passes through all three observations and has zero loss.


In [ ]:
x_lab8 = np.array([0.0, 1.0, 2.0])
y_lab8 = np.array([1.0, 3.0, 5.0])

def loss_lab8(theta):
    w, b = theta
    return np.mean((w*x_lab8+b-y_lab8)**2)

def gradient_lab8(theta):
    w, b = theta
    residuals = w*x_lab8+b-y_lab8
    return np.array([2*np.mean(x_lab8*residuals), 2*np.mean(residuals)])

path_lab8 = [np.array([0.0, 0.0])]
eta_lab8 = 0.1
for _ in range(60):
    current = path_lab8[-1]
    path_lab8.append(current-eta_lab8*gradient_lab8(current))
path_lab8 = np.array(path_lab8)

w_grid_lab8 = np.linspace(-0.5, 3.2, 250)
b_grid_lab8 = np.linspace(-0.5, 2.8, 250)
W_grid_lab8, B_grid_lab8 = np.meshgrid(w_grid_lab8, b_grid_lab8)
L_grid_lab8 = np.zeros_like(W_grid_lab8)
for x_value, y_value in zip(x_lab8, y_lab8):
    L_grid_lab8 += (W_grid_lab8*x_value+B_grid_lab8-y_value)**2/len(x_lab8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].contour(W_grid_lab8, B_grid_lab8, L_grid_lab8, levels=20, cmap="viridis")
axes[0].plot(path_lab8[:, 0], path_lab8[:, 1], color="crimson", marker="o",
             markersize=3, label="training path")
axes[0].scatter([2], [1], marker="*", color="black", s=150, label="exact optimum")
axes[0].set_xlabel("slope w")
axes[0].set_ylabel("intercept b")
axes[0].set_title("Optimization in parameter space")
axes[0].legend()
axes[1].scatter(x_lab8, y_lab8, s=70, label="data")
x_line_lab8 = np.linspace(-0.2, 2.2, 200)
axes[1].plot(x_line_lab8, path_lab8[-1, 0]*x_line_lab8+path_lab8[-1, 1],
             color="crimson", label="trained line")
axes[1].plot(x_line_lab8, 2*x_line_lab8+1, linestyle="--", color="black",
             label="exact line")
axes[1].set_xlabel("input x")
axes[1].set_ylabel("target or prediction")
axes[1].set_title("Model-space view")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Final parameters:", path_lab8[-1])
print("Final loss:", loss_lab8(path_lab8[-1]))


## Integrated lab tasks

1. Verify the two partial-derivative formulas by expanding the loss first.
2. Compute the gradient at $$(w,b)=(0,0).$$
3. Compute the first gradient-descent update with $$\eta=0.1.$$
4. Construct the constant Hessian and show it is positive definite.
5. Explain the difference between the contour plot and the fitted-line plot.


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
At $$(0,0),$$

$$
L_w(0,0)=\frac{2}{3}(0+0-13)=-\frac{26}{3},
$$

and

$$
L_b(0,0)=2(0+0-3)=-6.
$$

Therefore

$$
\nabla L(0,0)=
\begin{bmatrix}
-26/3\\
-6
\end{bmatrix}.
$$

The first update is

$$
\begin{bmatrix}
w_1\\
b_1
\end{bmatrix}
=
\begin{bmatrix}
0\\
0
\end{bmatrix}
-0.1
\begin{bmatrix}
-26/3\\
-6
\end{bmatrix}
=
\boxed{
\begin{bmatrix}
13/15\\
3/5
\end{bmatrix}}.
$$

Differentiate the gradient again:

$$
H_L=
\begin{bmatrix}
10/3&2\\
2&2
\end{bmatrix}.
$$

Its leading entry is positive and its determinant is

$$
\frac{10}{3}(2)-2^2
=\frac{20}{3}-4
=\frac{8}{3}>0.
$$

Thus the Hessian is positive definite and the critical point is the unique global minimum.

The contour plot uses parameter coordinates and shows equal-loss pairs. The fitted-line plot uses data coordinates and shows what one parameter pair predicts as a function of the input.
""")


---

# Module 8 Summary

1. Multivariable functions map vectors to scalar or vector outputs.
2. Partial derivatives freeze all but one coordinate.
3. The gradient collects partial derivatives and points toward steepest local increase.
4. Directional derivatives are gradient dot products with unit directions.
5. The tangent plane is a first-order local approximation.
6. The multivariable chain rule multiplies along paths and adds across branches.
7. A Jacobian represents first derivatives of a vector-valued function.
8. A Hessian represents second-order curvature of a scalar function.
9. Vector gradient descent updates all parameters from one gradient evaluation.

## Cumulative terminology

| AI language | Multivariable-calculus interpretation |
|---|---|
| Feature vector | Point in an input space |
| Parameter vector | Point in a parameter space |
| Loss landscape | Scalar function over parameter space |
| Parameter gradient | Vector of loss sensitivities |
| Backpropagation | Efficient multivariable chain-rule computation |
| Saddle point | Zero-gradient point with rising and falling directions |
| Training step | Movement opposite the loss gradient |

## Exit ticket

1. What is held fixed in a partial derivative?
2. Why must a directional derivative use a unit direction when interpreted per unit distance?
3. Why does the negative gradient point toward steepest local decrease?
4. Why are derivative contributions added when a parameter reaches the loss through two branches?
5. Why can a zero gradient fail to identify a minimum?
